In [ ]:
# NFPA: Next-Frame Prediction Attack for AI Watermark Removal

import os
import random
import torch
import numpy as np
from PIL import Image
from tqdm import tqdm
import torchvision.transforms as transforms

from utils import MyStableDiffusionPipeline, TreeRingWatermark
from diffusers import DDIMScheduler, DDIMInverseScheduler

# Random seed for reproducibility
def set_seed(seed=1234):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

SEED = 1234
set_seed(SEED)

# ============================================================
# Configuration — Please modify the following paths as needed
# ============================================================
device = "cuda" if torch.cuda.is_available() else "cpu"

# Initialize NFP Attack Pipeline
model_id_or_path = "stabilityai/stable-diffusion-2-1-base"
pipe = MyStableDiffusionPipeline.from_pretrained(model_id_or_path, torch_dtype=torch.float16)

pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(device)
pipe.safety_checker = None
pipe.vae.requires_grad_(False)
pipe.text_encoder.requires_grad_(False)
pipe.unet.requires_grad_(False)

print("NFP Pipeline initialized")

In [ ]:
# Initialize Tree-Ring Watermark System
# Please modify the following COCO dataset paths to your local paths
COCO_ROOT = "./data/coco2017/val2017"
COCO_INSTANCE_ANN = "./data/coco2017/annotations/instances_val2017.json"
COCO_CAPTION_ANN = "./data/coco2017/annotations/captions_val2017.json"
WATERMARK_PATH = "./watermarks/stable_watermark.obj"
TREERING_BATCH_SIZE = 1

set_seed(SEED)
treering = TreeRingWatermark(
    checkpoint=model_id_or_path,
    device=device,
    image_size=512,
    watermark_channel=3,
    ring_radius=10,
    batch_size=TREERING_BATCH_SIZE,
    coco_root=COCO_ROOT,
    coco_instance_ann=COCO_INSTANCE_ANN,
    coco_caption_ann=COCO_CAPTION_ANN,
    watermark_path=WATERMARK_PATH,
)
print("TreeRing initialized")


# NFP Attack Functions (consistent with ai-watermark-attack-pipeline)
def inversion_latents_fun(image, num_inference_steps=10, custom_timesteps=None):
    """Perform DDIM inversion to get latent representation."""
    if custom_timesteps is not None:
        custom_timesteps = custom_timesteps[::-1].copy()
    
    if isinstance(image, Image.Image):
        image = transforms.ToTensor()(image).unsqueeze(0).to(device)
    elif isinstance(image, torch.Tensor):
        if len(image.shape) == 3:
            image = image.unsqueeze(0)
        image = image.to(device)
    
    image = image.half() * 2 - 1  # [0,1] -> [-1,1]
    
    latents = pipe.vae.encode(image)['latent_dist'].mean
    latents = latents * pipe.vae.config.scaling_factor
    pipe.scheduler = DDIMInverseScheduler.from_config(pipe.scheduler.config)
    inversion_latents = pipe(prompt="", negative_prompt="", num_inference_steps=num_inference_steps, custom_timesteps=custom_timesteps,
                             latents=latents, output_type='latent', width=512, height=512).images
    pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
    return inversion_latents, latents

In [ ]:
def nfp_attack(watermarked_image, num_inference_steps=10, xy=40, return_tensor=False):
    """Perform NFP attack on a watermarked image."""
    inversed_latents, _ = inversion_latents_fun(watermarked_image, num_inference_steps=num_inference_steps)
    
    warped_latents_timestep = torch.tensor([0], dtype=torch.long, device=pipe.device)
    images = pipe(
        prompt="", 
        num_images_per_prompt=2,
        latents=inversed_latents, 
        xyz=[xy, xy, 0],
        num_inference_steps=num_inference_steps, 
        warped_latents_timestep=warped_latents_timestep
    ).images
    
    attacked_image = images[1]
    if return_tensor:
        return transforms.ToTensor()(attacked_image)
    return attacked_image


# Configuration
NUM_IMAGES = 1000
START_IDX = 0

os.makedirs('./results/tree_ring/original', exist_ok=True)
os.makedirs('./results/tree_ring/watermarked', exist_ok=True)
os.makedirs('./results/tree_ring/attacked', exist_ok=True)

print(f"Processing {NUM_IMAGES} images starting from index {START_IDX}")

In [ ]:
# Step 1: Generate Tree-Ring Watermarked Images
num_images = NUM_IMAGES
original_images = []
watermarked_images = []
captions_list = []

set_seed(SEED)
treering.reset_iterator()

orig_imgs, wm_imgs, captions = treering.generate_watermarked_images_from_coco(
    num_images=num_images,
    start_idx=START_IDX
)

orig_tensor = orig_imgs
wm_tensor = wm_imgs

for idx in tqdm(range(num_images), desc="Saving images"):
    orig_pil = transforms.ToPILImage()(orig_imgs[idx].cpu())
    wm_pil = transforms.ToPILImage()(wm_imgs[idx].cpu())
    
    orig_pil.save(f'./results/tree_ring/original/{idx+1}.png')
    wm_pil.save(f'./results/tree_ring/watermarked/{idx+1}.png')
    
    original_images.append(orig_pil)
    watermarked_images.append(wm_pil)
    captions_list.append(captions[idx])

print(f"✓ Generated {num_images} image pairs")


In [ ]:
# Step 2: Watermark Detection (Before Attack)
print("Step 2: Detection Before Attack")

orig_scores = treering(orig_tensor)
wm_scores = treering(wm_tensor)

print(f"Original scores: mean={orig_scores.mean():.2f}")
print(f"Watermarked scores: mean={wm_scores.mean():.2f}")

TARGET_FPR = 0.01
threshold_at_fpr = treering.compute_threshold_at_fpr(orig_scores, TARGET_FPR)
tpr_watermarked, _, fpr_actual = treering.compute_tpr_at_fpr(wm_scores, orig_scores, TARGET_FPR)

print(f"\nBefore Attack (FPR={TARGET_FPR*100:.0f}%):")
print(f"  Threshold: {threshold_at_fpr:.2f}")
print(f"  TPR: {tpr_watermarked*100:.1f}%")


In [ ]:
# Step 3: NFP Attack
print("Step 3: NFP Attack")

NFP_INFERENCE_STEPS = 10
NFP_XY = 40
print(f"Parameters: steps={NFP_INFERENCE_STEPS}, xy={NFP_XY}")

attacked_tensor_list = []
attacked_orig_tensor_list = []

os.makedirs('./results/tree_ring/attacked', exist_ok=True)
os.makedirs('./results/tree_ring/attacked_orig', exist_ok=True)

for idx in tqdm(range(num_images), desc="Attacking watermarked"):
    attacked = nfp_attack(wm_tensor[idx], num_inference_steps=NFP_INFERENCE_STEPS, xy=NFP_XY, return_tensor=True)
    attacked_tensor_list.append(attacked)
    transforms.ToPILImage()(attacked).save(f'./results/tree_ring/attacked/{idx+1}.png')

for idx in tqdm(range(num_images), desc="Attacking original"):
    attacked_orig = nfp_attack(orig_tensor[idx], num_inference_steps=NFP_INFERENCE_STEPS, xy=NFP_XY, return_tensor=True)
    attacked_orig_tensor_list.append(attacked_orig)
    transforms.ToPILImage()(attacked_orig).save(f'./results/tree_ring/attacked_orig/{idx+1}.png')

attacked_tensor = torch.stack(attacked_tensor_list).to(device)
attacked_orig_tensor = torch.stack(attacked_orig_tensor_list).to(device)

print(f"✓ Attack completed")


In [ ]:
# Step 4: Detection After Attack
print("Step 4: Detection After Attack")

attacked_orig_scores = treering(attacked_orig_tensor)
attacked_scores = treering(attacked_tensor)

print(f"Attacked original scores: mean={attacked_orig_scores.mean():.2f}")
print(f"Attacked watermarked scores: mean={attacked_scores.mean():.2f}")

threshold_at_fpr_attacked = treering.compute_threshold_at_fpr(attacked_orig_scores, TARGET_FPR)
tpr_attacked = np.mean(attacked_scores < threshold_at_fpr_attacked)

print(f"\nAfter Attack (FPR={TARGET_FPR*100:.0f}%):")
print(f"  Threshold: {threshold_at_fpr_attacked:.2f}")
print(f"  TPR: {tpr_attacked*100:.1f}%")


In [ ]:
# Step 5: Final Summary
print("=" * 50)
print("FINAL RESULTS")
print("=" * 50)

threshold_final = threshold_at_fpr_attacked
tpr_attacked_final = np.mean(attacked_scores < threshold_final)

print(f"\nTPR@FPR={TARGET_FPR*100:.0f}% Results:")
print(f"  Before Attack: {tpr_watermarked*100:.1f}%")
print(f"  After Attack:  {tpr_attacked_final*100:.1f}%")
print(f"\n✓ Results saved to ./results/tree_ring/")


In [ ]:
# Optional: Visualize Results
import matplotlib.pyplot as plt

def visualize_results(original, watermarked, attacked, idx=0, save_path=None):
    """Visualize original, watermarked, and attacked images."""
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    
    def to_np(img):
        return img.cpu().permute(1, 2, 0).numpy() if isinstance(img, torch.Tensor) else img
    
    axes[0].imshow(to_np(original))
    axes[0].set_title(f'Original ({orig_scores[idx]:.1f})')
    axes[0].axis('off')
    
    axes[1].imshow(to_np(watermarked))
    axes[1].set_title(f'Watermarked ({wm_scores[idx]:.1f})')
    axes[1].axis('off')
    
    axes[2].imshow(to_np(attacked))
    axes[2].set_title(f'Attacked ({attacked_scores[idx]:.1f})')
    axes[2].axis('off')
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

for i in range(min(3, num_images)):
    visualize_results(orig_tensor[i], wm_tensor[i], attacked_tensor[i], idx=i,
                      save_path=f'./results/tree_ring/comparison_{i+1}.png')
